# Feature Engeneering ( Weight&biases) 

In [2]:
import wandb

In [3]:
wandb.login()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

  ········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\jijun\_netrc
wandb: Currently logged in as: nikoloz-kurua (giorgi-kvinikadze-univeristy) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

## Initialize Weights & Biases

In [29]:
wandb.init(project='hospital_readmission', name='feature_engineering_3')

## Load dataset

In [30]:
data_path = "processed_data.csv"
df = pd.read_csv(data_path)

## Feature Engineering: Creating interaction features

In [31]:
df['num_medications_changes'] = df.groupby('patient_nbr')['change'].transform('sum')
df['num_prior_visits'] = df.groupby('patient_nbr')['encounter_id'].transform('count')

## Encoding categorical variables using frequency encoding

In [32]:
categorical_features = ['diag_1', 'diag_2', 'diag_3']
for col in categorical_features:
    freq_map = df[col].value_counts(normalize=True).to_dict()
    df[col + '_freq'] = df[col].map(freq_map)

## Polynomial feature expansion

In [33]:
poly = PolynomialFeatures(degree=3, interaction_only=True, include_bias=False)
num_features = ['num_medications_changes', 'num_prior_visits']
poly_features = poly.fit_transform(df[num_features])
df_poly = pd.DataFrame(poly_features, columns=poly.get_feature_names_out(num_features))
df = pd.concat([df, df_poly], axis=1)


## Dimensionality reduction using PCA

In [34]:
pca = PCA(n_components=10)
pca_features = pca.fit_transform(df.select_dtypes(include=[np.number]))
df_pca = pd.DataFrame(pca_features, columns=[f'PCA_{i}' for i in range(10)])
df = pd.concat([df, df_pca], axis=1)

## Clustering-based feature creation

In [35]:
kmeans = KMeans(n_clusters=5, random_state=42)
df['cluster'] = kmeans.fit_predict(df.select_dtypes(include=[np.number]))

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


## Log dataset details to W&B

In [36]:
wandb.log({"num_samples": len(df), "num_features": len(df.columns)})

## Save processed data

In [37]:
df.to_csv("processed_data_engineered_3.csv", index=False)

In [38]:
wandb.finish()

num_features,▁
num_samples,▁
num_features,66
num_samples,101766


### After analyzing different versions, processed_data_engineered is best. 